# OCR Pipeline tổng hợp cho 3 quyển sách

Notebook này đã được refactor để chỉ đổi cấu hình ở đầu, còn pipeline xử lý nằm trong các hàm riêng.

## 1. Import thư viện

In [ ]:
# ===== IMPORTS =====
import os
import json
import time
import base64
import ast
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import requests

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


## 2. Cấu hình chạy

Đổi `BOOK` và `PAGE_NUMBER` ở cell dưới đây. Các đường dẫn, tham số OCR, file output nằm trong `CONFIGS`.

In [ ]:
# ===== CHỌN SÁCH + CẤU HÌNH =====
# Chỉ cần đổi BOOK và PAGE_NUMBER ở đây.
BOOK = "Chỉ Nam Ngọc Âm Giải Nghĩa"  # "Chinh Phụ Ngâm", "Truyền Kỳ Mạn Lục", "Chỉ Nam Ngọc Âm Giải Nghĩa"
PAGE_NUMBER = "014"                  # ví dụ: "014", "165", "page_172"

CONFIGS = {
    "Chinh Phụ Ngâm": {
        "input_dir": "./images",
        "output_dir": "./outputs/chinh_phu_ngam",
        "image_ext": ".png",
        "ocr_id": 1,
        "lang_type": 2,
        "reading_direction": 2,
        "excel_name": "ChinhPhuNgam.xlsx",
    },

    "Truyền Kỳ Mạn Lục": {
        "input_dir": "./images",
        "output_dir": "./outputs/truyen_ky_man_luc",
        "image_ext": ".png",
        "excel_name": "TruyenKyManLuc.xlsx",
        "llama_result_type": "markdown",
    },

    "Chỉ Nam Ngọc Âm Giải Nghĩa": {
        "input_dir": "./images",
        "output_dir": "./outputs/chi_nam_ngoc_am",
        "image_ext": ".jpg",
        "json_ext": ".json",
        "small_char_dim": (67, 90.75),
        "box_csv_name": "boxes.csv",
        "result_csv_name": "pipeline_results.csv",
        "task_slice": None,   # ví dụ debug 1 dòng: slice(69, 70). Chạy hết thì để None.
    },
}

cfg = CONFIGS[BOOK]
INPUT_DIR = Path(cfg["input_dir"])
OUTPUT_DIR = Path(cfg["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("BOOK:", BOOK)
print("PAGE_NUMBER:", PAGE_NUMBER)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 3. Hàm phụ trợ chung

In [ ]:
# ===== HÀM PHỤ TRỢ CHUNG =====
def page_path(page_number, ext, input_dir=INPUT_DIR):
    return str(input_dir / f"{page_number}{ext}")


def append_dataframe_to_excel(df, excel_path):
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    if excel_path.exists():
        old_df = pd.read_excel(excel_path)
        df = pd.concat([old_df, df], ignore_index=True)

    df.to_excel(excel_path, index=False, engine="openpyxl")
    print(f"Đã lưu Excel: {excel_path} ({len(df)} dòng tổng)")


def append_dataframe_to_csv(df, csv_path):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    file_exists = csv_path.exists()
    df.to_csv(csv_path, mode="a", index=False, header=not file_exists, encoding="utf-8-sig")
    print(f"Đã ghi CSV: {csv_path}")


# I. API CLC Sino-Nom

In [ ]:
def Image_upload(image_path):
    url_upload = "https://kimhannom.fit.hcmus.edu.vn/api/web/clc-sinonom/image-upload"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36 Edg/138.0.0.0",
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive"
    }
    
    with open(image_path, 'rb') as f:
        files = {'image_file': ('image.png', f, 'image/png')}
        response = requests.post(url_upload, headers=headers, files=files)
    
    if not response.text:
        print(f"Error: Empty response (Status: {response.status_code})")
        return None
    
    try:
        data = response.json()
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}\nStatus: {response.status_code}\nResponse: {response.text[:200]}")
        return None
    
    if data.get('is_success') and data.get('code') == '000000':
        file_name = data['data']['file_name']
        print(f"Done File: {file_name}")
        return file_name
    
    print(f"Error uploading image:\n  Code: {data.get('code')}\n  Message: {data.get('message')}")
    return None

In [ ]:
def Image_classification(image_path_server):
    url_classification = "https://kimhannom.fit.hcmus.edu.vn/api/web/clc-sinonom/image-classification"

    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36 Edg/138.0.0.0",
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Content-Type": "application/json"
    }

    data = {"file_name": image_path_server}

    classification_response = requests.post(url_classification, headers=headers, json=data)

    if not classification_response.text:
        print(f"Error: Empty response (Status: {classification_response.status_code})")
        return None

    try:
        data = classification_response.json()
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}\nStatus: {classification_response.status_code}\nResponse: {classification_response.text[:200]}")
        return None

    if data.get('is_success') and data.get('code') == '000000':
        result = data['data']
        print(f"Done - OCR ID: {result.get('ocr_id')}, Lang: {result.get('lang_type')}, Direction: {result.get('reading_direction')}")
        return result
    
    print(f"Error in classification:\n  Code: {data.get('code')}\n  Message: {data.get('message')}")
    return None

In [ ]:
import requests
import pandas as pd
import json

def Image_OCR(image_path_server, ocr_id, lang_type, reading_direction):
    url_ocr = "https://kimhannom.fit.hcmus.edu.vn/api/web/clc-sinonom/image-ocr"

    
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36 Edg/138.0.0.0",
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Content-Type": "application/json"
    }

    data = {
        "ocr_id": ocr_id,                    
        "file_name": image_path_server,
        "lang_type": lang_type,                
        "epitaph": 0,
        "reading_direction": reading_direction          
    }

    
    try:
        ocr_response = requests.post(url_ocr, headers=headers, json=data)
    except Exception as e:
        print(f"Connection Error: {e}")
        return None

    if not ocr_response.text:
        print(f"Error: Empty response (Status: {ocr_response.status_code})")
        return None

    try:
        data_resp = ocr_response.json()
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}\nStatus: {ocr_response.status_code}")
        return None

    if not data_resp.get('is_success') or data_resp.get('code') != '000000':
        print(f"Error in OCR API:\n Code: {data_resp.get('code')}\n Message: {data_resp.get('message')}")
        return None

    ocr = data_resp['data']['result_bbox']
    print(f"Done. Found {len(ocr)} characters.")

    df = pd.DataFrame({'box': [i[0] for i in ocr], 'ocr_char': [i[1][0] for i in ocr]})
    
    def get_min_max_h(a):
        temp = [i[0] for i in a]
        return min(temp), max(temp)

    def get_min_max_w(a):
        temp = [i[1] for i in a]
        return min(temp), max(temp)

    def is_in_range(min_max, c):
        return min_max[0] <= c <= min_max[1]

    def sort_box(df):
        results = {}
        midpoints = []

        for i in range(len(df)):
            min_max_w = get_min_max_h(df.loc[i, 'box'])
            c = sum(min_max_w) / 2  
            for j, midpoint in enumerate(midpoints):
                if is_in_range(min_max_w, midpoint):
                    results.setdefault(j, []).append(df.loc[i, 'box'])
                    break
            else: 
                midpoints.append(c)
                results[len(midpoints) - 1] = [df.loc[i, 'box']]

        if len(results) > 1:
            for i in range(len(results) - 1):
                for j in range(i + 1, len(results)):
                    temp_1 = results[i][0]
                    temp_2 = results[j][0]
                    max_1 = get_min_max_h(temp_1)[-1]
                    max_2 = get_min_max_h(temp_2)[-1]
                    if max_1 < max_2:
                        results[i], results[j] = results[j], results[i]

        sorted_boxes = []
        for i in range(len(results)):
            sorted_boxes.extend(sorted(results[i], key=lambda x: get_min_max_w(x)[-1], reverse=False))

        final_data = []
        for box in sorted_boxes:
            matching_rows = df[df['box'].apply(lambda x: list(x) == list(box))]
            if not matching_rows.empty:
                final_data.append({'box': box, 'ocr_char': matching_rows['ocr_char'].values[0]})

        return pd.DataFrame(final_data)
    
    return sort_box(df)

In [ ]:
def Sinonom_Transliteration(text, font_type, lang_type):
    """
        lang_type 0: unk, 1: Sino, 2: Nom
    """

    url_transliteration = "https://kimhannom.fit.hcmus.edu.vn/api/web/clc-sinonom/sinonom-transliteration"

    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36 Edg/138.0.0.0",
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Content-Type": "application/json"
    }

    data = {
        "text": text,
        "font_type": font_type, 
        "lang_type": lang_type   
    }

    transliteration_response = requests.post(url_transliteration, headers=headers, json=data)

    if not transliteration_response.text:
        print("Error: Empty response from server")
        print(f"Status code: {transliteration_response.status_code}")
        return None

    try:
        data = json.loads(transliteration_response.text)
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        print(f"Status code: {transliteration_response.status_code}")
        print(f"Response text: {transliteration_response.text[:200]}")
        return None

    if data.get('is_success') and data.get('code') == '000000':
        results_text = data['data']['result_text_transcription']
        return results_text
    else:
        print("Error in transliteration:")
        print(f"  Code: {data.get('code')}")
        print(f"  Message: {data.get('message')}")
        return None



In [ ]:
def Sinonom_Fair_Seg(text, lang_type):
    url_fair_seg = "https://kimhannom.fit.hcmus.edu.vn/api/web/clc-sinonom/sinonom-fair-seg"

    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36 Edg/138.0.0.0",
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Content-Type": "application/json"
    }

    data = {"text": text, "lang_type": lang_type}

    fair_seg_response = requests.post(url_fair_seg, headers=headers, json=data)

    if not fair_seg_response.text:
        print(f"Error: Empty response (Status: {fair_seg_response.status_code})")
        return None

    try:
        data = fair_seg_response.json()
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}\nStatus: {fair_seg_response.status_code}\nResponse: {fair_seg_response.text[:200]}")
        return None

    if data.get('is_success') and data.get('code') == '000000':
        print("Done - Fair segmentation completed")
        return data['data']

    print(f"Error in fair segmentation:\n  Code: {data.get('code')}\n  Message: {data.get('message')}")
    return None

In [ ]:
def Sinonom_Verse_Translation(text, font_type, lang_type):
    url_verse_translation = "https://kimhannom.fit.hcmus.edu.vn/api/web/clc-sinonom/sinonom-verse-translation"  

    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36 Edg/138.0.0.0",
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Content-Type": "application/json"
    }

    data = {"text": text, "font_type": font_type, "lang_type": lang_type}

    verse_translation_response = requests.post(url_verse_translation, headers=headers, json=data)

    if not verse_translation_response.text:
        print(f"Error: Empty response (Status: {verse_translation_response.status_code})")
        return None

    try:
        data = verse_translation_response.json()
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}\nStatus: {verse_translation_response.status_code}\nResponse: {verse_translation_response.text[:200]}")
        return None

    if data.get('is_success') and data.get('code') == '000000':
        print("Done - Verse translation completed")
        return data['data']['result']

    print(f"Error in verse translation:\n  Code: {data.get('code')}\n  Message: {data.get('message')}")
    return None

In [ ]:
def Sinonom_Administration_Translation(text, font_type, lang_type):
    """Translate Sino-Nom administration text. lang_type: 0=chưa biết, 1=Hán, 2=Nôm"""
    url_admin_translation = "https://kimhannom.fit.hcmus.edu.vn/api/web/clc-sinonom/sinonom-administration-translation"

    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36 Edg/138.0.0.0",
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Content-Type": "application/json"
    }

    data = {"text": text, "font_type": font_type, "lang_type": lang_type}

    admin_translation_response = requests.post(url_admin_translation, headers=headers, json=data)

    if not admin_translation_response.text:
        print(f"Error: Empty response (Status: {admin_translation_response.status_code})")
        return None

    try:
        data = admin_translation_response.json()
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}\nStatus: {admin_translation_response.status_code}\nResponse: {admin_translation_response.text[:200]}")
        return None

    if data.get('is_success') and data.get('code') == '000000':
        print("Done - Administration translation completed")
        return data['data']['result']

    print(f"Error in administration translation:\n  Code: {data.get('code')}\n  Message: {data.get('message')}")
    return None

## 4. Hiển thị OCR, bounding box và cắt ảnh

In [ ]:
def display_ocr_comparison(image_path, df, current_idx=None, text_result=""):
    img_original = Image.open(image_path)
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for idx, row in df.iterrows():
        box = row['box']
        pts = np.array(box, np.int32)
        pts = pts.reshape((-1, 1, 2))
        
        color = (255, 0, 0) if idx == current_idx else (0, 255, 0)
        thickness = 3 if idx == current_idx else 2
        
        cv2.polylines(img_rgb, [pts], True, color, thickness)

    fig, axes = plt.subplots(1, 2, figsize=(16, 10))

    axes[0].imshow(img_original)
    axes[0].set_title("Ảnh gốc")
    axes[0].axis('off')

    axes[1].imshow(img_rgb)
    axes[1].set_title(f"Kết quả OCR: {text_result}")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show() 

In [ ]:
import numpy as np

def sort_points_polygon(pts):
    """Sắp xếp 4 điểm: Top-Left, Top-Right, Bottom-Right, Bottom-Left."""
    pts = np.array(pts, dtype=np.float32)
    pts = pts[np.argsort(pts[:, 1])]
    top_pts = pts[:2][np.argsort(pts[:2, 0])] 
    bot_pts = pts[2:][np.argsort(pts[2:, 0])] 
    
    tl, tr = top_pts[0], top_pts[1]
    bl, br = bot_pts[0], bot_pts[1]
    return tl, tr, br, bl

def expand_segment(pA, pB, expand_ratio=0.05):
    """Nới rộng đoạn thẳng AB từ tâm ra 2 phía theo tỷ lệ."""
    center = (pA + pB) / 2.0
    v_half = (pB - pA) / 2.0
    v_half_exp = v_half * (1 + expand_ratio)
    return center - v_half_exp, center + v_half_exp

def estimate_small_columns(large_col_box, small_char_dim, num_chars, width_expansion=0.05):
    """
    Tính toán 2 cột tiểu tự nằm PHÍA DƯỚI cột đại tự.
    Chia đôi chiều rộng cột đại tự để làm 2 cột tiểu tự.
    """
    pts = np.array(large_col_box, dtype=np.float32)
    tl, tr, br, bl = sort_points_polygon(pts)
    
    v_down = ((bl - tl) + (br - tr)) / 2.0
    norm = np.linalg.norm(v_down)
    u_down = v_down / norm if norm > 0 else np.array([0, 1])
    
    mid_bottom = (bl + br) / 2.0
    
    h_small = num_chars * small_char_dim[1]
    margin_y = 5 
    
    start_base_L = bl + u_down * margin_y
    start_base_C = mid_bottom + u_down * margin_y
    start_base_R = br + u_down * margin_y
    
    rt_tl, rt_tr = expand_segment(start_base_C, start_base_R, width_expansion)
    rt_bl = rt_tl + u_down * h_small
    rt_br = rt_tr + u_down * h_small
    rt_column = [rt_tl, rt_tr, rt_br, rt_bl]

    lt_tl, lt_tr = expand_segment(start_base_L, start_base_C, width_expansion)
    lt_bl = lt_tl + u_down * h_small
    lt_br = lt_tr + u_down * h_small
    lt_column = [lt_tl, lt_tr, lt_br, lt_bl]
    
    rt_column = [[float(round(p[0])), float(round(p[1]))] for p in rt_column]
    lt_column = [[float(round(p[0])), float(round(p[1]))] for p in lt_column]
    
    return rt_column, lt_column

def estimate_small_columns_above(large_col_box, small_char_dim, num_chars, width_expansion=0.05):
    """
    Tính toán 2 cột tiểu tự nằm PHÍA TRÊN cột đại tự.
    """
    pts = np.array(large_col_box, dtype=np.float32)
    tl, tr, br, bl = sort_points_polygon(pts)

    v_down = ((bl - tl) + (br - tr)) / 2.0
    norm = np.linalg.norm(v_down)
    u_down = v_down / norm if norm > 0 else np.array([0, 1])
    
    mid_top = (tl + tr) / 2.0

    h_small = num_chars * small_char_dim[1]
    margin_y = 5

    end_base_L = tl - u_down * margin_y
    end_base_C = mid_top - u_down * margin_y
    end_base_R = tr - u_down * margin_y
    
    rt_bl, rt_br = expand_segment(end_base_C, end_base_R, width_expansion)
    rt_tl = rt_bl - u_down * h_small
    rt_tr = rt_br - u_down * h_small
    rt_col = [rt_tl, rt_tr, rt_br, rt_bl]
    
    lt_bl, lt_br = expand_segment(end_base_L, end_base_C, width_expansion)
    lt_tl = lt_bl - u_down * h_small
    lt_tr = lt_br - u_down * h_small
    lt_col = [lt_tl, lt_tr, lt_br, lt_bl]
    
    rt_col = [[float(round(p[0])), float(round(p[1]))] for p in rt_col]
    lt_col = [[float(round(p[0])), float(round(p[1]))] for p in lt_col]
    
    return rt_col, lt_col

In [ ]:
import numpy as np

def sort_points_polygon_2(pts):
    """Sắp xếp 4 điểm: Top-Left, Top-Right, Bottom-Right, Bottom-Left."""
    pts = np.array(pts, dtype=np.float32)
    pts = pts[np.argsort(pts[:, 1])]
    top_pts = pts[:2][np.argsort(pts[:2, 0])] 
    bot_pts = pts[2:][np.argsort(pts[2:, 0])] 
    
    tl, tr = top_pts[0], top_pts[1]
    bl, br = bot_pts[0], bot_pts[1]
    return tl, tr, br, bl

def expand_segment_2(pA, pB, expand_ratio=0.05):
    """Nới rộng đoạn thẳng AB từ tâm ra 2 phía theo tỷ lệ."""
    center = (pA + pB) / 2.0
    v_half = (pB - pA) / 2.0
    v_half_exp = v_half * (1 + expand_ratio)
    return center - v_half_exp, center + v_half_exp

def estimate_small_columns_2(large_col_box, small_char_dim, num_chars, width_expansion=0.05):
    """
    Tính toán 1 box tiểu tự DUY NHẤT nằm PHÍA DƯỚI cột đại tự (bao trùm cả 2 cột).
    """
    pts = np.array(large_col_box, dtype=np.float32)
    tl, tr, br, bl = sort_points_polygon(pts)

    v_down = ((bl - tl) + (br - tr)) / 2.0
    norm = np.linalg.norm(v_down)
    u_down = v_down / norm if norm > 0 else np.array([0, 1])

    h_small = num_chars * small_char_dim[1]
    margin_y = 5 
    
    start_L = bl + u_down * margin_y
    start_R = br + u_down * margin_y
    
    box_tl, box_tr = expand_segment_2(start_L, start_R, width_expansion)
    
    box_bl = box_tl + u_down * h_small
    box_br = box_tr + u_down * h_small
    
    box_column = [box_tl, box_tr, box_br, box_bl]
    
    box_column = [[float(round(p[0])), float(round(p[1]))] for p in box_column]
    
    return box_column

def estimate_small_columns_above_2(large_col_box, small_char_dim, num_chars, width_expansion=0.05):
    """
    Tính toán 1 box tiểu tự DUY NHẤT nằm PHÍA TRÊN cột đại tự.
    """
    pts = np.array(large_col_box, dtype=np.float32)
    tl, tr, br, bl = sort_points_polygon(pts)
    
    v_down = ((bl - tl) + (br - tr)) / 2.0
    norm = np.linalg.norm(v_down)
    u_down = v_down / norm if norm > 0 else np.array([0, 1])
    
    h_small = num_chars * small_char_dim[1]
    margin_y = 5
    
    end_L = tl - u_down * margin_y
    end_R = tr - u_down * margin_y
    
    box_bl, box_br = expand_segment(end_L, end_R, width_expansion)
    
    box_tl = box_bl - u_down * h_small
    box_tr = box_br - u_down * h_small
    
    box_column = [box_tl, box_tr, box_br, box_bl]
    
    box_column = [[float(round(p[0])), float(round(p[1]))] for p in box_column]
    
    return box_column

In [ ]:

def read_bbx_1(json_path, image_path, small_char_dim):
    print(f"--- BẮT ĐẦU XỬ LÝ TRANG: {image_path} ---")
    
    with open(json_path, 'r', encoding='utf-8') as f:
        label_data = json.load(f)
        
    tasks = []
    
    for i, shape in enumerate(label_data['shapes']):
        label = shape['label']
        if 'dai_tu' not in label:
            continue
            
        dai_tu_box = shape['points']
        dai_tu_box = [[float(round(pt[0])), float(round(pt[1]))] for pt in dai_tu_box]
        
        desc = shape.get("description", "").strip()
        parts = desc.split()
        
        num_below = 0
        num_above = 0

        if len(parts) >= 1 and parts[0].isdigit():
            num_below = int(parts[0])
        if len(parts) >= 2 and parts[1].isdigit():
            num_above = int(parts[1])
            
        if num_above > 0:
            rt_col_tren, lt_col_tren = estimate_small_columns_above(
                dai_tu_box, small_char_dim, num_above
            )
            tasks.append({'id': f"{label}_tieu_tu_tren_phai", 'type': 'tieu_tu', 'box': rt_col_tren})
            tasks.append({'id': f"{label}_tieu_tu_tren_trai", 'type': 'tieu_tu', 'box': lt_col_tren})
        
        tasks.append({
            'id': label,
            'type': 'dai_tu',
            'box': dai_tu_box
        })
        
        if num_below > 0:
            rt_col_duoi, lt_col_duoi = estimate_small_columns(
                dai_tu_box, small_char_dim, num_below
            )
            rt_col_duoi = [[float(round(pt[0])), float(round(pt[1]))] for pt in rt_col_duoi]
            lt_col_duoi = [[float(round(pt[0])), float(round(pt[1]))] for pt in lt_col_duoi]
            
            tasks.append({'id': f"{label}_tieu_tu_duoi_phai", 'type': 'tieu_tu', 'box': rt_col_duoi})
            tasks.append({'id': f"{label}_tieu_tu_duoi_trai", 'type': 'tieu_tu', 'box': lt_col_duoi})
            
    df_tasks = pd.DataFrame(tasks)
    print(f"Đã lên danh sách {len(df_tasks)} khối chữ cần lưu.\n")
    return df_tasks

In [ ]:
import json
import pandas as pd

def read_bbx_2(json_path, image_path, small_char_dim):
    print(f"--- BẮT ĐẦU XỬ LÝ TRANG: {image_path} ---")
    
    with open(json_path, 'r', encoding='utf-8') as f:
        label_data = json.load(f)
        
    tasks = []
    
    for i, shape in enumerate(label_data['shapes']):
        label = shape['label']
        if 'dai_tu' not in label:
            continue
            
        dai_tu_box = shape['points']
        dai_tu_box = [[float(round(pt[0])), float(round(pt[1]))] for pt in dai_tu_box]
     
        desc = shape.get("description", "").strip()
        parts = desc.split()
        
        num_below = 0
        num_above = 0
        
        if len(parts) >= 1 and parts[0].isdigit():
            num_below = int(parts[0])
        if len(parts) >= 2 and parts[1].isdigit():
            num_above = int(parts[1])
            
        if num_above > 0:
            col_tren = estimate_small_columns_above_2(dai_tu_box, small_char_dim, num_above)
            tasks.append({'id': f"{label}_tieu_tu_tren", 'type': 'tieu_tu', 'box': col_tren})
        
        tasks.append({
            'id': label,
            'type': 'dai_tu',
            'box': dai_tu_box
        })
        
        if num_below > 0:
            col_duoi = estimate_small_columns_2(dai_tu_box, small_char_dim, num_below)
            tasks.append({'id': f"{label}_tieu_tu_duoi", 'type': 'tieu_tu', 'box': col_duoi})
            
    df_tasks = pd.DataFrame(tasks)
    print(f"Đã lên danh sách {len(df_tasks)} khối chữ cần đọc.\n")
    return df_tasks

In [ ]:

import cv2
import numpy as np

def warp_perspective_crop(image_path, box, output_path="./crop.png", border_size=5):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Không thể đọc ảnh từ: {image_path}")
        return

    pts = np.array(box, dtype=np.float32)
    
    # Tính toán kích thước khối sau khi warp
    width_A = np.linalg.norm(pts[0] - pts[1])
    width_B = np.linalg.norm(pts[2] - pts[3])
    max_width = max(int(width_A), int(width_B))
    
    height_A = np.linalg.norm(pts[0] - pts[3])
    height_B = np.linalg.norm(pts[1] - pts[2])
    max_height = max(int(height_A), int(height_B))
    
    dst_pts = np.array([
        [0, 0],
        [max_width - 1, 0],
        [max_width - 1, max_height - 1],
        [0, max_height - 1]
    ], dtype=np.float32)

    M = cv2.getPerspectiveTransform(pts, dst_pts)
    warped_cropped = cv2.warpPerspective(img, M, (max_width, max_height))

    warped_with_border = cv2.copyMakeBorder(
        warped_cropped, 
        top=border_size, 
        bottom=border_size, 
        left=border_size, 
        right=border_size, 
        borderType=cv2.BORDER_CONSTANT, 
        value=[0, 0, 0]
    )
    
    cv2.imwrite(output_path, warped_with_border)
    return warped_with_border

## 5. Fallback OpenRouter

In [ ]:
try:
    from openai import OpenAI
except Exception:
    OpenAI = None

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
client = None

if OpenAI is not None and OPENROUTER_API_KEY:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )
else:
    print("OpenRouter chưa được cấu hình. Nếu CLC API trả rỗng, fallback sẽ trả chuỗi rỗng.")


def ocr_with_openrouter(image_path, prompt=None, model="qwen/qwen3-vl-32b-instruct"):
    if client is None:
        print("Không có OPENROUTER_API_KEY, bỏ qua fallback OpenRouter.")
        return ""

    if prompt is None:
        prompt = "Đọc tất cả ký tự trong ảnh. Đây là văn bản Hán-Nôm. Chỉ trả về văn bản, không giải thích."

    with open(image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    try:
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
                        },
                    ],
                }
            ],
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"OpenRouter error: {e}")
        return ""


## 6. OCR 

In [ ]:
def process_single_block_with_fallback(image_path, box_coords, block_id, lang_type, force_openrouter_ids=None):
    """
    Cắt ảnh theo box, gọi CLC OCR. Nếu CLC rỗng hoặc block_id nằm trong force_openrouter_ids,
    tự động gọi OpenRouter fallback.
    """
    force_openrouter_ids = set(force_openrouter_ids or [])
    crop_filename = str(OUTPUT_DIR / f"temp_{block_id}.png")
    text_result = ""
    img_crop = None

    try:
        warp_perspective_crop(image_path, box_coords, output_path=crop_filename)
        img_crop = Image.open(crop_filename).convert("RGB")

        print(f"  [{block_id}] Đang gọi CLC API...")
        path_server = Image_upload(crop_filename)

        if path_server:
            df_ocr = Image_OCR(path_server, ocr_id=1, lang_type=lang_type, reading_direction=1)
            if df_ocr is not None and not df_ocr.empty:
                text_result = "".join(df_ocr["ocr_char"].astype(str).tolist())

        if (not text_result.strip()) or (block_id in force_openrouter_ids):
            print(f"  [{block_id}] CLC API rỗng hoặc bị ép fallback. Chuyển sang OpenRouter...")
            text_result = ocr_with_openrouter(crop_filename)
            text_result = text_result.replace("```json", "").replace("```", "").strip()

    except Exception as e:
        print(f"  [{block_id}] Có lỗi xảy ra: {e}")

    finally:
        if os.path.exists(crop_filename):
            os.remove(crop_filename)

    return text_result, img_crop


def run_blocks_ocr(image_path, df_tasks, output_csv, sleep_short=20, sleep_long=30, force_openrouter_ids=None):
    final_results = []
    total_tasks = len(df_tasks)

    print(f"\n>> BẮT ĐẦU PIPELINE OCR CHO {total_tasks} KHỐI CHỮ...")

    for index, (orig_idx, row) in enumerate(df_tasks.iterrows()):
        block_id = row["id"]
        block_type = row["type"]
        box_coords = row["box"]

        print(f"\n[{index + 1}/{total_tasks}] Đang xử lý: {block_id} ({block_type})")

        lang_type = 2 if "tieu" in block_type else 1
        text_result, img_obj = process_single_block_with_fallback(
            image_path=image_path,
            box_coords=box_coords,
            block_id=block_id,
            lang_type=lang_type,
            force_openrouter_ids=force_openrouter_ids,
        )

        display_ocr_comparison(image_path, df_tasks, current_idx=orig_idx, text_result=text_result)
        print(f"  -> VĂN BẢN ĐỌC ĐƯỢC: {text_result}")

        final_results.append({
            "ID": block_id,
            "Type": block_type,
            "Box": box_coords,
            "Text": text_result,
        })

        if index < total_tasks - 1:
            wait_time = sleep_short if index < 25 else sleep_long
            print(f"Đang nghỉ {wait_time} giây...")
            time.sleep(wait_time)

    df_final = pd.DataFrame(final_results)
    display(df_final)
    append_dataframe_to_csv(df_final, output_csv)
    return df_final


## 7. PIPELINE FOR EACH BOOK

In [ ]:
def run_chinh_phu_ngam(page_number=PAGE_NUMBER, cfg=cfg):
    image_path = page_path(page_number, cfg["image_ext"])
    excel_path = OUTPUT_DIR / cfg["excel_name"]

    path_server = Image_upload(image_path)
    if not path_server:
        print("Upload ảnh thất bại.")
        return None

    df = Image_OCR(
        image_path_server=path_server,
        ocr_id=cfg["ocr_id"],
        lang_type=cfg["lang_type"],
        reading_direction=cfg["reading_direction"],
    )

    if df is None:
        print("OCR thất bại.")
        return None

    display_ocr_comparison(image_path, df)
    append_dataframe_to_excel(df, excel_path)
    return df


def run_truyen_ky_man_luc(page_number=PAGE_NUMBER, cfg=cfg):
    try:
        from llama_parse import LlamaParse
    except Exception as e:
        print("Chưa cài hoặc chưa import được LlamaParse. Hãy cài llama-parse trước.")
        print(e)
        return None

    api_key = os.getenv("LLAMA_CLOUD_API_KEY")
    if not api_key:
        print("Thiếu LLAMA_CLOUD_API_KEY. Hãy set biến môi trường trước khi chạy.")
        return None

    image_path = page_path(page_number, cfg["image_ext"])
    md_path = OUTPUT_DIR / f"{page_number}.md"
    excel_path = OUTPUT_DIR / cfg["excel_name"]

    parser = LlamaParse(
        api_key=api_key,
        result_type=cfg.get("llama_result_type", "markdown"),
        extract_charts=True,
        auto_mode=True,
        auto_mode_trigger_on_image_in_page=True,
        auto_mode_trigger_on_table_in_page=True,
    )

    try:
        documents = parser.load_data(image_path)
    except Exception as e:
        print(f"Lỗi LlamaParse: {e}")
        return None

    if not documents:
        print("Không có dữ liệu từ LlamaParse.")
        return None

    with open(md_path, "w", encoding="utf-8") as f:
        for doc in documents:
            XXX
            print(doc.text)
    print(f"Saved markdown: {md_path}")

    rows = []
    with open(md_path, "r", encoding="utf-8") as f:
        for line in f:
            clean_line = line.strip()
            if clean_line:
                rows.append({
                    "ID": page_number,
                    "Image Box": "",
                    "SinoNom OCR": clean_line,
                    "Chữ Quốc Ngữ": "",
                })

    df = pd.DataFrame(rows)
    append_dataframe_to_excel(df, excel_path)
    return df


def run_chi_nam_ngoc_am(page_number=PAGE_NUMBER, cfg=cfg):
    image_path = page_path(page_number, cfg["image_ext"])
    json_path = page_path(page_number, cfg["json_ext"])
    small_char_dim = cfg["small_char_dim"]

    box_csv = OUTPUT_DIR / cfg["box_csv_name"]
    result_csv = OUTPUT_DIR / cfg["result_csv_name"]

    print(">> ĐANG SINH DANH SÁCH TỌA ĐỘ...")
    df_tasks_save = read_bbx_1(json_path, image_path, small_char_dim)
    append_dataframe_to_csv(df_tasks_save, box_csv)
    display_ocr_comparison(image_path=image_path, df=df_tasks_save)

    df_tasks = read_bbx_2(json_path, image_path, small_char_dim)

    task_slice = cfg.get("task_slice")
    if task_slice is not None:
        df_tasks = df_tasks.iloc[task_slice]

    df_final = run_blocks_ocr(
        image_path=image_path,
        df_tasks=df_tasks,
        output_csv=result_csv,
        force_openrouter_ids=cfg.get("force_openrouter_ids", []),
    )
    return df_final


## 8. RUN

In [ ]:

if BOOK == "Chinh Phụ Ngâm":
    df_result = run_chinh_phu_ngam()

elif BOOK == "Truyền Kỳ Mạn Lục":
    df_result = run_truyen_ky_man_luc()

elif BOOK == "Chỉ Nam Ngọc Âm Giải Nghĩa":
    df_result = run_chi_nam_ngoc_am()

else:
    raise ValueError(f"BOOK không hợp lệ: {BOOK}")
